# 技能5 · Day 5 上机：用 LangSmith + tiktoken 生产化营销 Agent

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **LangSmith** `@traceable` 为营销 Agent 配置端到端追踪，查看每步 token/延迟/工具调用
2. 用 **tiktoken** 精确统计 token 成本，结合模型定价计算单次和日均成本
3. 设计**延迟监控**方案：分步计时，识别 P50/P95 瓶颈
4. 实现**灾备降级**（多级 fallback）和 **CI/CD**（pytest 回归测试 + 评估门禁）
5. 用**压测**模拟并发请求，观察延迟/成本/成功率

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：langsmith（LangChain 可观测性平台 SDK）+ tiktoken（OpenAI token 计数）。
营销映射：将营销内容生成 Agent 从 PoC 推向生产环境。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ langsmith 云端 trace 上传需要 `LANGSMITH_API_KEY`（可选，不设则本地追踪不上传）。
> tiktoken 是纯本地库，无需 API key。

In [ ]:
# !pip install langsmith tiktoken -q
# 可选：启用 LangSmith 云端 trace 上传
# export LANGSMITH_TRACING=true
# export LANGSMITH_API_KEY=lsv2_sk_...
# export LANGSMITH_PROJECT=marketing-agent-prod

## 1. 场景背景与营销映射

**生产化对象**：营销内容生成 Agent（生成小红书种草文案/朋友圈广告）。

**PoC 状态**：Agent 能跑通，但：
- 不知道每次请求消耗多少 token / 花多少钱
- 不知道哪个步骤慢（知识库检索 vs LLM 推理）
- 主模型 API 故障时系统直接崩溃
- 代码修改后没有回归测试，可能引入质量问题
- 不知道高并发下系统表现如何

**本上机解决**：用真实生产级工具（langsmith + tiktoken）逐一解决上述问题。

| TODO | 生产化维度 | 工具 | 解决的问题 |
|------|-----------|------|-----------|
| TODO1 | 可观测性 | LangSmith @traceable | 追踪每步调用链 |
| TODO2 | 成本控制 | tiktoken | 精确计 token + 算成本 |
| TODO3 | 延迟优化 | time.perf_counter | 分步计时，识别瓶颈 |
| TODO4 | 灾备降级 | ResilientLLM | 多级 fallback |
| TODO5 | CI/CD | pytest + GitHub Actions | 回归测试 + 评估门禁 |
| TODO6 | 压测 | ThreadPoolExecutor | 并发性能测试 |

**模拟环境说明**：本上机用模拟 LLM 函数（离线可运行），实际使用时替换为真实 OpenAI/Anthropic API 调用。tiktoken 的 token 计数和成本计算是真实的。

In [ ]:
import os
import time
import json
import logging

# ============================================================
# 营销 Agent 模拟环境（离线可运行）
# 实际使用时替换为真实 LLM API 和产品数据库
# ============================================================

# 产品知识库（模拟）
PRODUCT_KB = {
    "烟酰胺精华液": {
        "name": "烟酰胺亮肤精华液",
        "ingredients": "5%烟酰胺",
        "effects": "提亮肤色、收缩毛孔",
        "texture": "清爽水润",
        "price": "199元/30ml",
        "target": "25-35岁女性"
    },
    "丝绒口红": {
        "name": "丝绒哑光口红",
        "ingredients": "哑光质地",
        "effects": "持久不脱色",
        "texture": "丝绒哑光",
        "price": "198元",
        "target": "20-40岁女性"
    },
    "防晒霜": {
        "name": "清透防晒霜",
        "ingredients": "SPF50+ PA++++",
        "effects": "防晒黑",
        "texture": "轻薄透气",
        "price": "159元/50ml",
        "target": "所有肤质"
    },
}

# 营销 Brief 集
MARKETING_BRIEFS = [
    "为烟酰胺精华液写小红书种草文案，目标人群25-35岁女性",
    "为丝绒口红写朋友圈广告",
    "为防晒霜写小红书种草文案",
    "为烟酰胺精华液写朋友圈广告",
    "为防晒霜写朋友圈广告",
]

# 模型定价表（$/token，基于 OpenAI 2026 定价）
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
}

def mock_search_kb(query: str) -> dict:
    """模拟知识库搜索（实际使用时替换为向量检索）"""
    for key, val in PRODUCT_KB.items():
        if key in query:
            return val
    return {}

def mock_generate_content(brief: str, product_info: dict) -> str:
    """模拟 LLM 生成营销文案（离线可运行）。
    实际使用时替换为：openai.ChatCompletion.create(...) 或 langchain LLM 调用。
    """
    if not product_info:
        return f"抱歉，未找到相关产品信息。Brief: {brief}"
    name = product_info.get("name", "产品")
    effects = product_info.get("effects", "优质")
    price = product_info.get("price", "价格优惠")
    return (
        f"姐妹们！这款{name}真的绝了 "
        f"主打{effects}，效果看得见！"
        f"价格才{price}，性价比拉满！"
        f"评论区扣1获取链接 #{name}"
    )

def mock_llm_primary(prompt: str) -> str:
    """模拟主模型 gpt-4o"""
    product_info = mock_search_kb(prompt)
    return mock_generate_content(prompt, product_info)

def mock_llm_backup(prompt: str) -> str:
    """模拟备用模型 gpt-4o-mini（输出略短）"""
    product_info = mock_search_kb(prompt)
    if not product_info:
        return "未找到产品信息"
    return f"推荐{product_info['name']}，{product_info['effects']}，{product_info['price']}。"

print("环境初始化完成")
print(f"产品知识库: {len(PRODUCT_KB)} 个产品")
print(f"营销Brief: {len(MARKETING_BRIEFS)} 条")
print(f"模型定价: {list(MODEL_PRICING.keys())}")

## TODO 1：配置 LangSmith 追踪，运行营销 Agent

**LangSmith** 是 LangChain 出品的 LLM 可观测性平台。核心 API：
- `@traceable`：装饰器，自动追踪函数调用链（输入/输出/延迟/嵌套）
- `wrap_openai`：包装 OpenAI client，自动记录 LLM 调用
- `Client`：程序化查询 trace 数据（`list_runs`）

即使不配置 `LANGSMITH_API_KEY`，`@traceable` 仍会在本地记录调用链。配置 API key 后可在 https://smith.langchain.com 查看可视化 trace。

In [ ]:
import os

# 配置 LangSmith 追踪（无 API key 时本地追踪不上传，但 @traceable 仍工作）
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "marketing-agent-prod")
# os.environ["LANGSMITH_API_KEY"] = "lsv2_sk_..."  # 实际使用时设置

from langsmith import traceable, Client

@traceable(name="营销内容生成Agent")
def marketing_agent(brief: str) -> str:
    """营销内容生成 Agent：搜索知识库 -> 生成文案"""
    product_info = mock_search_kb(brief)
    content = mock_generate_content(brief, product_info)
    return content

# 运行 Agent
result = marketing_agent("为烟酰胺精华液写小红书种草文案，目标人群25-35岁女性")
print(f"Agent 输出：{result}")

# 查看 trace（如果配置了 API key，可查询云端 trace）
try:
    client = Client()
    runs = list(client.list_runs(project_name="marketing-agent-prod", limit=3))
    print(f"\n最近 {len(runs)} 条 trace：")
    for run in runs:
        print(f"  - {run.name}: status={run.status}, latency={run.total_seconds():.3f}s")
    trace_result = {"agent_output": result, "trace_count": len(runs)}
except Exception as e:
    print(f"\n（未配置 LANGSMITH_API_KEY，无法查询云端 trace: {type(e).__name__}）")
    print("Agent 仍正常运行，@traceable 在本地记录了调用链。")
    print("配置 API key 后可在 https://smith.langchain.com 查看可视化 trace。")
    trace_result = {"agent_output": result, "trace_count": 0, "note": "Set LANGSMITH_API_KEY to enable cloud trace"}

print(f"\nAgent 输出: {trace_result}")

## TODO 2：用 tiktoken 精确统计 Token 成本

**tiktoken** 是 OpenAI 的 BPE 分词器，比按字符估算精确得多。

**成本计算公式**：
- input_cost = input_tokens × model_pricing["input"]
- output_cost = output_tokens × model_pricing["output"]
- total_cost = input_cost + output_cost

**为什么不能按字符估算**：中文一个字约 1-2 个 token，英文一个单词约 1-2 个 token，标点和 emoji 也消耗 token。只有用 tiktoken 精确计数才能算出真实成本。

In [ ]:
import tiktoken

def calculate_token_cost(prompt: str, response: str, model: str = "gpt-4o") -> dict:
    """精确计算 LLM 调用的 token 成本"""
    enc = tiktoken.encoding_for_model(model)
    input_tokens = len(enc.encode(prompt))
    output_tokens = len(enc.encode(response))

    rates = MODEL_PRICING.get(model, MODEL_PRICING["gpt-4o"])
    input_cost = input_tokens * rates["input"]
    output_cost = output_tokens * rates["output"]
    total_cost = input_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

# 计算单次请求成本
sample_prompt = "为一款烟酰胺精华液写小红书种草文案，目标人群25-35岁女性"
sample_response = mock_generate_content(sample_prompt, mock_search_kb(sample_prompt))

cost_report = calculate_token_cost(sample_prompt, sample_response, "gpt-4o")

print("=== Token 成本报告 ===")
print(f"模型: {cost_report['model']}")
print(f"输入: {cost_report['input_tokens']} tokens (${cost_report['input_cost']:.6f})")
print(f"输出: {cost_report['output_tokens']} tokens (${cost_report['output_cost']:.6f})")
print(f"总计: {cost_report['total_tokens']} tokens (${cost_report['total_cost']:.6f})")
print(f"\n日均成本估算（10000次/天）: ${cost_report['total_cost'] * 10000:.2f}")
print(f"月均成本估算: ${cost_report['total_cost'] * 10000 * 30:.2f}")

# 对比：用 gpt-4o-mini 的成本
cost_mini = calculate_token_cost(sample_prompt, sample_response, "gpt-4o-mini")
print(f"\n对比 gpt-4o-mini: ${cost_mini['total_cost']:.6f}/次 (节省 {(1 - cost_mini['total_cost']/cost_report['total_cost'])*100:.1f}%)")

## 2. 延迟监控：识别 Agent 瓶颈

生产环境中用户期望 5 秒内响应。Agent 的总延迟是各步骤延迟的叠加：
- 知识库检索：通常 < 100ms（向量检索）
- LLM 推理：通常 1-10s（取决于模型和输出长度）
- 工具调用：取决于外部 API 响应时间

**优化策略**：
- 并行化：独立步骤并行执行
- 缓存：相似请求复用响应
- 模型路由：简单任务用小模型（gpt-4o-mini），复杂任务用大模型（gpt-4o）
- 流式输出：LLM 流式返回，用户感知延迟降低

## TODO 3：延迟监控 -- 分步计时，识别瓶颈

为 Agent 各步骤添加 `time.perf_counter()` 计时，多次运行后计算 P50/P95 延迟，识别瓶颈步骤。

In [ ]:
import time

def run_agent_with_timing(brief: str) -> dict:
    """运行 Agent 并记录每步延迟"""
    timings = {}

    # 步骤1：搜索知识库
    t0 = time.perf_counter()
    product_info = mock_search_kb(brief)
    timings["search_kb"] = time.perf_counter() - t0

    # 步骤2：生成内容
    t0 = time.perf_counter()
    content = mock_generate_content(brief, product_info)
    timings["generate_content"] = time.perf_counter() - t0

    # 总延迟
    timings["total"] = timings["search_kb"] + timings["generate_content"]

    return {"content": content, "timings": timings}

# 多次运行收集延迟数据
N_RUNS = 20
all_timings = []
for brief in (MARKETING_BRIEFS * 4)[:N_RUNS]:
    report = run_agent_with_timing(brief)
    all_timings.append(report["timings"])

# 计算 P50/P95
def percentile(data, p):
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return sorted_data[min(idx, len(sorted_data) - 1)]

steps = ["search_kb", "generate_content", "total"]
latency_report = {
    "n_runs": N_RUNS,
    "p50": {step: percentile([t[step] for t in all_timings], 50) for step in steps},
    "p95": {step: percentile([t[step] for t in all_timings], 95) for step in steps},
}

print(f"=== 延迟监控报告（{N_RUNS} 次运行）===")
for step in steps:
    p50 = latency_report["p50"][step] * 1000
    p95 = latency_report["p95"][step] * 1000
    print(f"  {step}: P50={p50:.2f}ms, P95={p95:.2f}ms")

# 识别瓶颈
bottleneck = max(steps[:-1], key=lambda s: latency_report["p50"][s])
print(f"\n瓶颈步骤: {bottleneck}（P50={latency_report['p50'][bottleneck]*1000:.2f}ms）")
print("优化建议: 若 generate_content 是瓶颈，考虑模型路由（简单任务用 gpt-4o-mini）")

## TODO 4：灾备降级 -- 多级 Fallback

生产环境中 LLM API 可能超时、限流、宕机。系统需要优雅降级：

```
主模型 (gpt-4o)
  ↓ 失败
备用模型 (gpt-4o-mini)
  ↓ 失败
默认模板（预设营销文案模板）
```

**关键设计**：
- 每个模型重试 2 次后切换
- 记录降级日志（哪个模型失败、何时切换）
- 最终降级返回预设模板（而非报错崩溃）

In [ ]:
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

class ResilientLLM:
    """具有多级 fallback 的 LLM 调用器"""

    def __init__(self):
        self.model_chain = [
            ("gpt-4o", mock_llm_primary),
            ("gpt-4o-mini", mock_llm_backup),
        ]
        self.max_retries = 2
        self.retry_delay = 0.05  # 秒

    def invoke_with_fallback(self, prompt: str) -> str:
        """带降级的 LLM 调用"""
        for model_name, llm_func in self.model_chain:
            for attempt in range(self.max_retries):
                try:
                    response = llm_func(prompt)
                    if model_name != self.model_chain[0][0]:
                        logger.info(f"降级使用模型: {model_name}")
                    return response
                except Exception as e:
                    logger.warning(f"{model_name} 调用失败 (attempt {attempt+1}): {e}")
                    time.sleep(self.retry_delay * (attempt + 1))
            logger.warning(f"{model_name} 全部重试失败，尝试下一个模型")

        logger.error("所有模型不可用，返回降级响应")
        return self.fallback_response(prompt)

    def fallback_response(self, prompt: str) -> str:
        """最终降级：返回预设模板"""
        return "【降级模板】产品名称：[待填充]。核心卖点：[待填充]。请稍后重试获取定制内容。"

resilient_llm = ResilientLLM()

# 测试1：正常调用
print("=== 正常调用 ===")
result = resilient_llm.invoke_with_fallback("为烟酰胺精华液写文案")
print(f"结果: {result[:80]}...")

# 测试2：模拟主模型故障，触发降级
print("\n=== 降级测试（模拟主模型故障）===")
def failing_llm(prompt: str) -> str:
    raise RuntimeError("API 超时（模拟）")

original_func = resilient_llm.model_chain[0][1]
resilient_llm.model_chain[0] = ("gpt-4o", failing_llm)
result = resilient_llm.invoke_with_fallback("为烟酰胺精华液写文案")
print(f"降级结果: {result[:80]}...")

# 测试3：所有模型故障，触发最终降级
print("\n=== 最终降级测试（所有模型故障）===")
resilient_llm.model_chain[1] = ("gpt-4o-mini", failing_llm)
result = resilient_llm.invoke_with_fallback("为烟酰胺精华液写文案")
print(f"最终降级结果: {result[:80]}...")

# 恢复
resilient_llm.model_chain[0] = ("gpt-4o", original_func)
resilient_llm.model_chain[1] = ("gpt-4o-mini", mock_llm_backup)
print("\n已恢复模型链。")
print(f"\nResilientLLM: {resilient_llm}")

## TODO 5：CI/CD -- 回归测试 + 评估门禁

Agent 系统的 CI/CD 比传统软件更复杂：输出非确定性，不能用精确断言。核心是**评估门禁**：

```
代码提交
  → [1] 代码质量检查（linting/类型检查）
  → [2] 单元测试（Mock LLM，不消耗 token）
  → [3] 集成测试（真实 LLM，检查输出格式/安全/质量）
  → [4] 评估门禁（通过率 >= 90%? 幻觉率 <= 5%? 安全违规 = 0%?）
  → [5] 部署 / 阻止部署
```

本 TODO 用 pytest 风格写回归测试，并模拟 GitHub Actions YAML。

In [ ]:
# CI/CD: Agent 回归测试 + 评估门禁
# 文件名: test_marketing_agent.py（pytest 风格）

test_cases = [
    {"input": "为烟酰胺精华液写小红书种草文案", "expect_keywords": ["烟酰胺", "精华"]},
    {"input": "为丝绒口红写朋友圈广告", "expect_keywords": ["口红", "丝绒"]},
    {"input": "为防晒霜写小红书种草文案", "expect_keywords": ["防晒"]},
]

# === 回归测试函数 ===

def test_agent_output_not_empty():
    """测试1: Agent 输出非空"""
    for case in test_cases:
        result = mock_generate_content(case["input"], mock_search_kb(case["input"]))
        assert result is not None and len(result) > 0, f"输入 '{case['input']}' 输出为空"

def test_agent_output_contains_keywords():
    """测试2: Agent 输出包含产品关键词"""
    for case in test_cases:
        result = mock_generate_content(case["input"], mock_search_kb(case["input"]))
        for kw in case["expect_keywords"]:
            assert kw in result, f"输入 '{case['input']}' 输出缺少关键词 '{kw}'"

def test_agent_output_no_forbidden_words():
    """测试3: Agent 输出不含违禁词（广告法合规）"""
    forbidden = ["最好", "第一", "国家级", "顶级"]
    for case in test_cases:
        result = mock_generate_content(case["input"], mock_search_kb(case["input"]))
        for word in forbidden:
            assert word not in result, f"输出包含违禁词 '{word}'"

# === 评估门禁 ===

def evaluate_ci_gate() -> dict:
    """运行所有测试，统计通过率，判断是否通过门禁"""
    tests = [test_agent_output_not_empty, test_agent_output_contains_keywords, test_agent_output_no_forbidden_words]
    passed = 0
    failed = 0
    for test in tests:
        try:
            test()
            passed += 1
            print(f"  PASS: {test.__name__}")
        except AssertionError as e:
            failed += 1
            print(f"  FAIL: {test.__name__}: {e}")

    gate_passed = (passed == len(tests)) and (passed / len(tests) >= 0.9)
    return {"passed": passed, "failed": failed, "total": len(tests), "gate_passed": gate_passed}

print("=== CI 评估门禁 ===")
ci_result = evaluate_ci_gate()
print(f"\n门禁结果: {ci_result['passed']}/{ci_result['total']} 通过", end="")
print(" -> 部署批准" if ci_result["gate_passed"] else " -> 阻止部署")

# === GitHub Actions YAML（模拟）===
gh_actions_yaml = """name: Marketing Agent CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Install dependencies
        run: pip install langsmith tiktoken pytest
      - name: Run agent regression tests
        run: pytest test_marketing_agent.py -v
      - name: Evaluation gate
        run: python evaluate_gate.py --min-pass-rate 0.90 --max-hallucination 0.05
"""
print("\n=== GitHub Actions YAML ===")
print(gh_actions_yaml)
print(f"\nCI 结果: {ci_result}")

## TODO 6：压测 -- 并发请求性能测试

生产环境可能面临高并发（如营销活动期间）。压测目标：
- 测量吞吐量（req/s）
- 测量 P50/P95 延迟
- 测量总成本
- 检测限流/超时/成功率下降

用 `concurrent.futures.ThreadPoolExecutor` 模拟 N 个并发用户同时请求。

In [ ]:
import concurrent.futures
import time

def single_request(brief: str) -> dict:
    """单次请求：运行 Agent + 计时 + 计成本"""
    t0 = time.perf_counter()
    product_info = mock_search_kb(brief)
    content = mock_generate_content(brief, product_info)
    latency = time.perf_counter() - t0

    # 计算 token 成本
    enc = tiktoken.encoding_for_model("gpt-4o")
    input_tokens = len(enc.encode(brief))
    output_tokens = len(enc.encode(content))
    cost = (input_tokens * MODEL_PRICING["gpt-4o"]["input"] +
            output_tokens * MODEL_PRICING["gpt-4o"]["output"])

    return {"latency": latency, "tokens": input_tokens + output_tokens, "cost": cost, "success": True}

# 压测配置
N_CONCURRENT = 20
briefs = (MARKETING_BRIEFS * (N_CONCURRENT // len(MARKETING_BRIEFS) + 1))[:N_CONCURRENT]

print(f"开始压测：{N_CONCURRENT} 并发请求...")
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_CONCURRENT) as executor:
    futures = [executor.submit(single_request, b) for b in briefs]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]
wall_time = time.perf_counter() - t0

# 统计
latencies = sorted([r["latency"] for r in results])
total_cost = sum(r["cost"] for r in results)
total_tokens = sum(r["tokens"] for r in results)

def percentile(data, p):
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return sorted_data[min(idx, len(sorted_data) - 1)]

stress_report = {
    "concurrent_requests": N_CONCURRENT,
    "wall_time": wall_time,
    "throughput": N_CONCURRENT / wall_time,
    "p50_latency": percentile(latencies, 50),
    "p95_latency": percentile(latencies, 95),
    "total_tokens": total_tokens,
    "total_cost": total_cost,
    "success_rate": sum(1 for r in results if r["success"]) / len(results),
}

print(f"\n=== 压测报告 ({N_CONCURRENT} 并发) ===")
print(f"  总耗时: {stress_report['wall_time']:.3f}s")
print(f"  吞吐量: {stress_report['throughput']:.1f} req/s")
print(f"  P50延迟: {stress_report['p50_latency']*1000:.2f}ms")
print(f"  P95延迟: {stress_report['p95_latency']*1000:.2f}ms")
print(f"  总token: {stress_report['total_tokens']}")
print(f"  总成本: ${stress_report['total_cost']:.6f}")
print(f"  成功率: {stress_report['success_rate']:.1%}")
print(f"\n  推算日均成本（此负载 24h）: ${stress_report['total_cost'] * (3600 / stress_report['wall_time']) * 24:.2f}")
print(f"\n压测报告: {stress_report}")

## 3. 反思与前沿

### 反思问题
1. 你的营销 Agent 在压测中表现出什么瓶颈？（延迟飙升 / 成本超预算 / 限流？）
2. 如果日均 10000 次请求，月成本是多少？是否可接受？如何优化（模型路由/缓存/自建 vLLM）？
3. 灾备降级中，主模型故障时 fallback 到 gpt-4o-mini，输出质量会下降吗？如何监控质量下降？
4. CI 门禁的通过率阈值（90%）是否合理？太高会导致什么问题？太低呢？

### 2026 前沿：推理成本优化
- **vLLM**（https://github.com/vllm-project/vllm）：PagedAttention + 连续批处理，吞吐量 14-24x，自建推理服务替代商业 API
- **投机解码**（arXiv 2211.17192）：小模型生成候选 token，大模型并行验证，延迟降低 2-3x
- **MoE**（arXiv 2401.04088）：Mixture of Experts，总参数大但单次推理计算量小，成本更低
- **LangGraph checkpointer**：Agent 中断恢复，服务重启时从 checkpoint 恢复，节省重复 token 消耗

参考 [vLLM](https://github.com/vllm-project/vllm) + [投机解码论文](https://arxiv.org/abs/2211.17192) + [DeepSeek-MoE](https://arxiv.org/abs/2401.04088)。